## Tutorial de Análise Avançada de Dados usando Amazon AgentCore Bedrock Code Interpreter (Langchain)
Este tutorial demonstra como criar um agente de IA que realiza análise avançada de dados através da execução de código usando Python. Utilizamos o Amazon Bedrock AgentCore Code Interpreter para executar código gerado pelo LLM.

Este tutorial demonstra como usar o AgentCore Bedrock Code Interpreter para:
1. Configurar um ambiente sandbox
2. Configurar um agente baseado em Langchain que realiza análise avançada de dados gerando código baseado na consulta do usuário
3. Executar código em um ambiente sandbox usando o Code Interpreter
4. Exibir os resultados de volta ao usuário

## Pré-requisitos
- Conta AWS com acesso ao Bedrock AgentCore Code Interpreter
- Você possui as permissões IAM necessárias para criar e gerenciar recursos do code interpreter
- Pacotes Python necessários instalados (incluindo boto3, bedrock-agentcore & langchain)
- A role IAM deve ter permissões para invocar modelos no Amazon Bedrock
 - Acesso ao modelo Claude 3.7 Sonnet na região US Oregon (us-west-2)

## Sua role de execução IAM deve ter a seguinte política IAM anexada

~~~ {
"Version": "2012-10-17",
"Statement": [
    {
        "Effect": "Allow",
        "Action": [
            "bedrock-agentcore:CreateCodeInterpreter",
            "bedrock-agentcore:StartCodeInterpreterSession",
            "bedrock-agentcore:InvokeCodeInterpreter",
            "bedrock-agentcore:StopCodeInterpreterSession",
            "bedrock-agentcore:DeleteCodeInterpreter",
            "bedrock-agentcore:ListCodeInterpreters",
            "bedrock-agentcore:GetCodeInterpreter"
        ],
        "Resource": "*"
    },
    {
        "Effect": "Allow",
        "Action": [
            "logs:CreateLogGroup",
            "logs:CreateLogStream",
            "logs:PutLogEvents"
        ],
        "Resource": "arn:aws:logs:*:*:log-group:/aws/bedrock-agentcore/code-interpreter*"
    }
]
}

## Como funciona

O sandbox de execução de código permite que os agentes processem consultas de usuários com segurança, criando um ambiente isolado com um interpretador de código, shell e sistema de arquivos. Após um Large Language Model auxiliar na seleção de ferramentas, o código é executado dentro desta sessão, antes de ser retornado ao usuário ou Agente para síntese.

![architecture local](code-interpreter.png)

## 1. Configurando o Ambiente

Primeiro, vamos importar as bibliotecas necessárias e inicializar nosso cliente Code Interpreter.

O timeout padrão da sessão é de 900 segundos (15 minutos). No entanto, iniciamos a sessão com uma duração de timeout ligeiramente maior, de 1200 segundos (20 minutos), já que realizaremos análises detalhadas em nossos dados

In [ ]:
!pip install --upgrade -r requirements.txt

In [ ]:
from bedrock_agentcore.tools.code_interpreter_client import CodeInterpreter
from langchain.agents import AgentExecutor, create_tool_calling_agent, initialize_agent, tool
from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
import json
import pandas as pd
from typing import Dict, Any, List

# Initialize the Code Interpreter within a supported AWS region.
code_client = CodeInterpreter('us-west-2')
code_client.start(session_timeout_seconds=1200)

## 2. Lendo o Arquivo de Dados Local

Agora vamos ler o conteúdo do nosso arquivo de dados de exemplo. O arquivo consiste em dados aleatórios com 4 colunas: Name, Preferred_City, Preferred_Animal, Preferred_Thing e aproximadamente 300.000 registros.

Iremos analisar este arquivo usando um agente um pouco mais tarde, para entender distribuições e outliers

In [ ]:
df_data = pd.read_csv("samples/data.csv")
df_data.head()

In [ ]:
def read_file(file_path: str) -> str:
    """Helper function to read file content with error handling"""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return ""
    except Exception as e:
        print(f"An error occurred: {e}")
        return ""

data_file_content = read_file("samples/data.csv")

## 3. Preparando Arquivos para o Ambiente Sandbox

Vamos criar uma estrutura que define os arquivos que queremos criar no ambiente sandbox.

In [ ]:
files_to_create = [
                {
                    "path": "data.csv",
                    "text": data_file_content
                }]

## 4. Criando Função Auxiliar para Invocação de Ferramenta

Esta função auxiliar facilitará a chamada de ferramentas do sandbox e o tratamento de suas respostas. Dentro de uma sessão ativa, você pode executar código em linguagens suportadas (Python, JavaScript), acessar bibliotecas baseadas em sua configuração de dependências, gerar visualizações e manter o estado entre execuções.

In [ ]:
def call_tool(tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    """Helper function to invoke sandbox tools

    Args:
        tool_name (str): Name of the tool to invoke
        arguments (Dict[str, Any]): Arguments to pass to the tool

    Returns:
        Dict[str, Any]: JSON formatted result
    """
    response = code_client.invoke(tool_name, arguments)
    for event in response["stream"]:
        return json.dumps(event["result"])

## 5. Escrever arquivo de dados no Code Sandbox

Agora vamos escrever nosso arquivo de dados no ambiente sandbox e verificar se foram criados com sucesso.

In [ ]:
# Write files to sandbox
writing_files = call_tool("writeFiles", {"content": files_to_create})
print("Writing files result:")
print(writing_files)

# Verify files were created
listing_files = call_tool("listFiles", {"path": ""})
print("\nFiles in sandbox:")
print(listing_files)

## 6. Realizar Análise Avançada usando Agente baseado em Langchain

Agora vamos configurar um agente para realizar análise de dados no arquivo de dados que enviamos para o sandbox (acima)

### 6.1 Definição do Prompt de Sistema
Definir o comportamento e capacidades do assistente de IA. Instruímos nosso assistente a sempre validar respostas através da execução de código e raciocínio baseado em dados.

In [ ]:
SYSTEM_PROMPT = """You are a helpful AI assistant that validates all answers through code execution using the tools provided. DO NOT Answer questions without using the tools

VALIDATION PRINCIPLES:
1. When making claims about code, algorithms, or calculations - write code to verify them
2. Use execute_python to test mathematical calculations, algorithms, and logic
3. Create test scripts to validate your understanding before giving answers
4. Always show your work with actual code execution
5. If uncertain, explicitly state limitations and validate what you can

APPROACH:
- If asked about a programming concept, implement it in code to demonstrate
- If asked for calculations, compute them programmatically AND show the code
- If implementing algorithms, include test cases to prove correctness
- Document your validation process for transparency
- The sandbox maintains state between executions, so you can refer to previous results

TOOL AVAILABLE:
- execute_python: Run Python code and see output

RESPONSE FORMAT: The execute_python tool returns a JSON response with:
- sessionId: The sandbox session ID
- id: Request ID
- isError: Boolean indicating if there was an error
- content: Array of content objects with type and text/data
- structuredContent: For code execution, includes stdout, stderr, exitCode, executionTime

For successful code execution, the output will be in content[0].text and also in structuredContent.stdout.
Check isError field to see if there was an error.

Be thorough, accurate, and always validate your answers when possible."""

### 6.2 Definição da Ferramenta de Execução de Código
A seguir, definimos a função como ferramenta que será usada pelo Agente como tool, para executar código no code sandbox. Usamos o decorador @tool para anotar a função como uma ferramenta personalizada para o Agente.

Dentro de uma sessão ativa do code interpreter, você pode executar código em linguagens suportadas (Python, JavaScript), acessar bibliotecas baseadas em sua configuração de dependências, gerar visualizações e manter o estado entre execuções.

In [ ]:
#Define and configure the code interpreter tool
@tool
def execute_python(code: str, description: str = "") -> str:
    """Execute Python code in the sandbox."""

    if description:
        code = f"# {description}\n{code}"

    #Print generated Code to be executed
    print(f"\n Generated Code: {code}")


    # Call the Invoke method and execute the generated code, within the initialized code interpreter session
    response = code_client.invoke("executeCode", {
        "code": code,
        "language": "python",
        "clearContext": False
    })
    for event in response["stream"]:
        return json.dumps(event["result"])

### 6.3 Configuração do Agente
Criamos e configuramos um agente usando o SDK Langchain. Fornecemos a ele o prompt de sistema e a ferramenta que definimos acima para executar código gerado

#### 6.4 Inicializar o modelo de linguagem

Para usar o modelo Claude Haiku 4.5, especificamos o [Cross-region Inference (CRIS)](https://docs.aws.amazon.com/bedrock/latest/userguide/cross-region-inference.html) profile id

In [ ]:
llm = ChatBedrockConverse(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",region_name="us-west-2")

#### 6.5 Definir o template de prompt

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

#### 6.6 Criar uma lista de nossas ferramentas personalizadas

In [ ]:
tools = [execute_python]

### 6.7 Criar o executor do agente

In [ ]:
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## 7. Invocação do Agente e Processamento de Resposta
Invocamos o agente com nossa consulta e processamos a resposta do agente


Nota: A execução assíncrona requer execução em um ambiente assíncrono

## 7.1 Consulta para realizar Análise Exploratória de Dados (EDA)

Vamos começar com uma consulta que instrui o agente a realizar análise exploratória de dados no arquivo de dados no ambiente code sandbox

In [ ]:
query = "Load the file 'data.csv' and perform exploratory data analysis(EDA) on it. Tell me about distributions and outlier values."

response=agent_executor.invoke({"input": query})
print("\n*********Final Results*********")
print(response['output'][0]['text'])

## 7.2 Consulta para extrair informação

Agora, vamos instruir o agente a extrair informações específicas do arquivo de dados no ambiente code sandbox

In [ ]:
query = "Within the file 'data.csv', how many individuals with the first name 'Kimberly' have 'Crocodile' as their favourite animal?"

response=agent_executor.invoke({"input": query})
print("\n*********Final Results*********")
print(response['output'][0]['text'])

## 8. Limpeza

Finalmente, vamos limpar parando a sessão do Code Interpreter. Assim que terminar de usar uma sessão, a sessão deve ser interrompida para liberar recursos e evitar cobranças desnecessárias.

In [ ]:
# Stop the Code Interpreter session
code_client.stop()
print("Code Interpreter session stopped successfully!")